In [0]:
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
table_bronze = dbutils.widgets.get("table_bronze")
table_silver = dbutils.widgets.get("table_silver")

In [0]:
df_bronze = spark.read.table(f"{catalog}.{schema_bronze}.{table_bronze}")

## Transform Data


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.dataframe import DataFrame

df_bronze = df_bronze.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

data_type_mapping = {
    "customer_id": "string",
    "customer_unique_id": "string",
    "customer_zip_code_prefix": "string",
    "customer_city": "string",
    "customer_state": "string"
}




def cast_columns(df: DataFrame, mapping: dict) -> DataFrame:
    existing_columns = df.columns

    for column, data_type in mapping.items():
        if column not in existing_columns:
            raise ValueError(f"Column '{column}' not found in DataFrame")
            

        df = df.withColumn(column, F.col(column).cast(data_type))
        print(f"Column '{column}' casted to {data_type}")
    
    return df

df_bronze_normalized = cast_columns(df_bronze, data_type_mapping)


In [0]:
from pyspark.sql import DataFrame

def standardardizing_column_name(df: DataFrame) -> DataFrame:
    new_columns = [column.strip().replace(" ", "_").lower() for column in df.columns]
    return df.toDF(*new_columns)

def standardarizing_raw(df: DataFrame) -> DataFrame:
    string_columns = [col_name for col_name, dtype in df.dtypes if dtype == 'string']

    for col in string_columns:
        df = df.withColumn(col, F.trim(F.lower(F.col(col))))
        
    return df
    

df_bronze_column_normalized = standardardizing_column_name(df_bronze_normalized)
df_bronze_standardized = standardarizing_raw(df_bronze_column_normalized)


## Silver

In [0]:
df_silver = df_bronze_standardized.fillna("nao_informado", subset=["customer_city", "customer_state", "customer_zip_code_prefix"])\
            .withColumn("silver_update_date", F.current_timestamp())